# Module 3: SCD Type 2 — Handling Boundary Changes

**Target audience:** SQL + Tableau analysts (no Python code required)

**Note:** This is a **READ-ONLY exploration notebook**. The SCD Type 2 DML (UPDATE, INSERT) was already executed during Docker initialization. We explore the results to understand how SCD Type 2 preserves historical accuracy.

In [ ]:
# Setup: Import the helper function
from ddc_helpers import run_query

## Scenario: Administrative Boundary Split

**March 2024:** The Thai government splits subdistrict **"บางเขน"** (Bang Khen, code 100901) in Chatuchak district, Bangkok, into TWO new subdistricts:

- **"บางเขนเหนือ"** (Bang Khen Nuea) — code **100901** (keeps the old code)
- **"บางเขนใต้"** (Bang Khen Tai) — code **100902** (new code)

### The Challenge

- **3 dengue cases from 2023** were diagnosed in the **OLD** Bang Khen boundary
- **1 dengue case from April 2024** happened in the **NEW** northern half (Bang Khen Nuea)
- DDC dashboards must show **BOTH** old and new data correctly

### Why NOT Just UPDATE the Old Row?

**Because that would rewrite history!**

- The old polygon covered a **different geographic area**
- 2023 dengue cases were **geocoded to the OLD boundary**
- If we overwrite it with the new boundaries, spatial queries on 2023 data give **wrong results**
- Historical analysis would be **permanently corrupted**

## The SCD Type 2 Approach (Conceptual Walkthrough)

Instead of updating, we use a **2-step process** to preserve history:

### Step 1: EXPIRE the Old Row
```sql
UPDATE dim_location
SET end_date   = '2024-02-29',  -- day before split effective date
    is_current = FALSE
WHERE subdistrict_code = '100901' AND is_current = TRUE;
```

- The old row is **NOT deleted**
- It remains in the table with `is_current = FALSE`
- Historical facts still join to it via `location_sk` (surrogate key)

### Step 2: INSERT New Versions
```sql
INSERT INTO dim_location (...)
VALUES
  ('100901', 'บางเขนเหนือ', ..., '2024-03-01', '9999-12-31', TRUE),  -- new northern half
  ('100902', 'บางเขนใต้', ..., '2024-03-01', '9999-12-31', TRUE);    -- new southern half
```

- Each new subdistrict gets its own **new `location_sk`** (via IDENTITY)
- `start_date = '2024-03-01'` (effective date of the split)
- `end_date = '9999-12-31'` (current version)
- `is_current = TRUE`

### Result: Complete Audit Trail

`dim_location` now has **3 rows** related to Bang Khen:

1. Old **"บางเขน"** (expired: 2023-01-01 to 2024-02-29) — `is_current = FALSE`
2. New **"บางเขนเหนือ"** (current: 2024-03-01 to 9999-12-31) — `is_current = TRUE`
3. New **"บางเขนใต้"** (current: 2024-03-01 to 9999-12-31) — `is_current = TRUE`

Historical facts still join via the old `location_sk` to the expired row. New facts join to the new rows.

## Verification 1: The Audit Trail

Let's query `dim_location` to see the 3 rows related to Bang Khen:

In [ ]:
# Query the SCD Type 2 audit trail for Bang Khen split
run_query("""
SELECT location_sk, subdistrict_code, subdistrict_name_th,
       start_date, end_date, is_current
FROM ddc_training.dim_location
WHERE subdistrict_code IN ('100901', '100902')
ORDER BY start_date, subdistrict_code
""")

**Expected output:**

| location_sk | subdistrict_code | subdistrict_name_th | start_date | end_date   | is_current |
|-------------|------------------|---------------------|------------|------------|-----------|
| 1           | 100901           | บางเขน              | 2023-01-01 | 2024-02-29 | FALSE     |
| 6           | 100901           | บางเขนเหนือ         | 2024-03-01 | 9999-12-31 | TRUE      |
| 7           | 100902           | บางเขนใต้           | 2024-03-01 | 9999-12-31 | TRUE      |

**Interpretation:**
- Row 1: The **old** Bang Khen boundary (expired on 2024-02-29)
- Row 6: The **new** Bang Khen Nuea (northern half, current as of 2024-03-01)
- Row 7: The **new** Bang Khen Tai (southern half, current as of 2024-03-01)

## Verification 2: Historical Facts Still Correct

Let's prove that **2023 dengue cases still link to the OLD Bang Khen boundary**:

In [ ]:
# Query 2023 infections linked to the OLD Bang Khen boundary
run_query("""
SELECT fi.patient_id,
       dd.full_date           AS infection_date,
       loc.subdistrict_name_th,
       loc.start_date         AS boundary_valid_from,
       loc.end_date           AS boundary_valid_until,
       loc.is_current         AS is_current_boundary,
       dis.disease_name_th
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
WHERE loc.subdistrict_code = '100901' AND dd.year_num = 2023
""")

**Expected output:**

- **3 rows** (PID-2023-00001, PID-2023-00002, PID-2023-00003)
- All show `subdistrict_name_th = 'บางเขน'` (the OLD name)
- All show `is_current_boundary = FALSE`
- `boundary_valid_from = 2023-01-01`, `boundary_valid_until = 2024-02-29`

**Interpretation:**
- The historical boundary is **preserved**
- 2023 data still references the **original undivided area**
- No data was lost or distorted
- Spatial queries on 2023 data use the **correct historical polygon**

## Verification 3: New Facts Link to New Boundaries

Let's verify that the **April 2024 case** now links to the **new Bang Khen Nuea boundary**:

In [ ]:
# Query the 2024 infection that was reassigned to the new northern boundary
run_query("""
SELECT fi.patient_id,
       dd.full_date           AS infection_date,
       loc.subdistrict_name_th,
       loc.start_date         AS boundary_valid_from,
       loc.is_current,
       dis.disease_name_th
FROM ddc_training.fact_infection fi
JOIN ddc_training.dim_date dd ON fi.infection_date_id = dd.date_id
JOIN ddc_training.dim_location loc ON fi.location_sk = loc.location_sk
JOIN ddc_training.dim_disease dis ON fi.disease_sk = dis.disease_sk
WHERE fi.patient_id = 'PID-2024-00001'
""")

**Expected output:**

- **1 row**: PID-2024-00001
- `subdistrict_name_th = 'บางเขนเหนือ'` (Bang Khen Nuea, the NEW northern half)
- `is_current = TRUE`
- `boundary_valid_from = 2024-03-01`

**Interpretation:**
- The 2024 case was **reassigned** to the correct new boundary
- DDC used geocoding (ST_Intersects) to determine which new polygon contains the patient's address
- The fact row's `location_sk` now points to the new boundary version

## Example 2: Rename Scenario (Simpler SCD Case)

The SQL also demonstrated a **rename** scenario for "หาดใหญ่" (Hat Yai) subdistrict.

Let's see the audit trail for that rename:

In [ ]:
# Query the SCD Type 2 audit trail for Hat Yai rename
run_query("""
SELECT location_sk, subdistrict_code, subdistrict_name_th,
       start_date, end_date, is_current
FROM ddc_training.dim_location
WHERE subdistrict_code = '900101'
ORDER BY start_date
""")

**Expected output:**

| location_sk | subdistrict_code | subdistrict_name_th | start_date | end_date   | is_current |
|-------------|------------------|---------------------|------------|------------|-----------|
| 5           | 900101           | หาดใหญ่             | 2023-01-01 | 2024-05-31 | FALSE     |
| 8           | 900101           | หาดใหญ่ (ปรับปรุง)  | 2024-06-01 | 9999-12-31 | TRUE      |

**Interpretation:**
- Same **2-step pattern**: expire old row, insert new version
- For renames, the **geometry stays the same** (copied from the old row)
- Only the name attribute changes

## Querying Current vs Historical Boundaries

When building dashboards or reports, you typically want to query **only current boundaries**:

In [ ]:
# Query all CURRENT locations (for use in dashboards)
run_query("""
SELECT location_sk, subdistrict_name_th, province_name_th, is_current
FROM ddc_training.dim_location
WHERE is_current = TRUE
ORDER BY province_name_th, subdistrict_name_th
LIMIT 10
""")

**Usage in dashboards:**

- **For current reports:** Add `WHERE is_current = TRUE`
- **For historical analysis:** Join via `location_sk` (no filter needed)
- **For point-in-time reports:** Add `WHERE '2023-06-01' BETWEEN start_date AND end_date`

## DDC Recommendation: Production Best Practices

### Use the 2-Step Approach for SPLIT/MERGE

```sql
BEGIN;  -- Always use transactions!

-- Step 1: Expire old row
UPDATE dim_location
SET end_date = '2024-02-29', is_current = FALSE
WHERE subdistrict_code = '100901' AND is_current = TRUE;

-- Step 2: Insert new versions
INSERT INTO dim_location (...) VALUES (...);

COMMIT;
```

### Why This Approach?

1. **Clearer** for code review and debugging
2. **Easier to audit** (each step can be verified independently)
3. **More flexible** (handles 1-to-many splits, many-to-1 merges)
4. **Transaction-safe** (COMMIT only if both steps succeed)

### Alternative: MERGE Statement

- Vertica supports `MERGE` (UPSERT), but with limitations:
  - `MERGE` can UPDATE old rows OR INSERT new rows, but NOT both in one statement
  - For SPLIT scenarios (1→N), you still need UPDATE + INSERT
  - Vertica CE does not support `MERGE` on tables with IDENTITY columns

**DDC recommendation:** Use the 2-step UPDATE + INSERT pattern for all SCD Type 2 operations.

## Exercise: Plan Your Own SCD Workflow

**Scenario:** Suppose **Khlong Toei** district (code 1018) in Bangkok splits into two new districts:

- **Khlong Toei North** (code 1018, keeps the old code)
- **Khlong Toei South** (code 1019, new code)

**Effective date:** 2025-01-01

**Question:** What SQL steps would you need to implement this split using SCD Type 2?

Write your conceptual steps below:

---

**Your notes here:**

1. 
2. 
3. 

---

**Hint:** Think about:
- Which table needs to be updated?
- What happens to the old row?
- How many new rows are inserted?
- What about existing fact records that reference the old district?

## Summary

**Key Takeaways:**

1. **SCD Type 2 preserves history** by keeping old versions of dimension rows
2. **Never UPDATE destructively** — always expire + insert new version
3. **Surrogate keys (location_sk) are critical** — they let facts join to the correct historical version
4. **Use `is_current` flag** for easy filtering in dashboards
5. **Use `start_date` and `end_date`** for point-in-time queries
6. **Always wrap SCD operations in transactions** to ensure consistency

**Real-world DDC use cases:**
- Administrative boundary changes (DOPA updates)
- Hospital name changes or mergers
- Disease classification updates (ICD-10 revisions)
- Population redistricting